# 01. EEG Emotion Recognition


## What This Notebook Does

This notebook follows the most basic machine learning flow:

1. load the EEG CSV file
2. split the data into `X` and `y`
3. scale the numeric features
4. train one SVM model
5. check accuracy
6. save the trained files


## Step 1: Setup

We load the main project folders so this notebook can find the dataset and save the trained model files.


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()
possible_dirs = [current_dir, current_dir / "NeuroSense" / "notebooks"]
notebooks_dir = next((path for path in possible_dirs if path.exists() and path.name == "notebooks"), None)
if notebooks_dir is None:
    raise FileNotFoundError("Start Jupyter from the project root or from NeuroSense/notebooks.")

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook()
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
RANDOM_STATE = ctx["random_state"]

print("Datasets directory:", DATASETS_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)


## Step 2: Import The Libraries

These are the only libraries we need for this very basic EEG notebook.


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC


## Step 3: Load The Dataset

`X` means input features.
`y` means the correct answer label.


In [ ]:
eeg_path = DATASETS_DIR / "eeg" / "eeg" / "emotions.csv"
if not eeg_path.exists():
    raise FileNotFoundError(f"Missing EEG dataset: {eeg_path}")

df = pd.read_csv(eeg_path)
print("Dataset shape:", df.shape)
print("First 5 labels:", df["label"].head().tolist())
df.head()


## Step 4: Prepare `X` And `y`

Here we separate the feature columns and the label column, then create a train/test split.


In [ ]:
if "label" not in df.columns:
    raise ValueError("The EEG CSV must contain a 'label' column.")

X = df.drop(columns=["label"]).select_dtypes(include="number").fillna(0.0)
y = df["label"].astype(str)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=RANDOM_STATE,
)

print("Feature matrix shape:", X.shape)
print("Class names:", list(encoder.classes_))


## Step 5: Scale The Features

SVM works better when numeric columns are on a similar scale.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## Step 6: Train The Model

We use one simple SVM classifier.


In [ ]:
model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)


## Step 7: Check The Result

We print the accuracy and also draw a confusion matrix.


In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
print("EEG test accuracy:", round(test_accuracy, 4))
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=encoder.classes_,
    cmap="Blues",
    xticks_rotation=20,
)
plt.title("EEG confusion matrix")
plt.tight_layout()
plt.show()


## Step 8: Save The Trained Files

The backend needs the same model, scaler, and label encoder later.


In [ ]:
artifact_dir = ARTIFACTS_DIR / "eeg"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "eeg_model.pkl")
joblib.dump(scaler, artifact_dir / "eeg_scaler.pkl")
joblib.dump(encoder, artifact_dir / "eeg_label_encoder.pkl")

metadata = {
    "data_source": "Public Dataset (DEAP/Kaggle-derived EEG CSV)",
    "data_source_note": "Tabular EEG features from the CSV used in this project.",
    "evaluation_method": "80/20 Stratified Random Split",
    "test_accuracy": round(float(test_accuracy), 4),
}
with open(artifact_dir / "eeg_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved EEG files to:", artifact_dir)
